In [ ]:
# --- Arranque del entorno local (en Google Colab no cambia nada) ---
import pathlib
import sys
import types

try:
    _raiz = next(
        d
        for d in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
        if (d / "curso_setup.py").exists()
    )
    sys.path.insert(0, str(_raiz))
    import curso_setup
except StopIteration:  # Google Colab: se usa un sustituto mínimo
    import subprocess

    def _clonar(destino="curso_IA_CHEC"):
        if not pathlib.Path(destino).is_dir():
            subprocess.run(
                ["git", "clone", "https://github.com/UN-GCPDS/curso_IA_CHEC.git", destino],
                check=True,
            )
        return pathlib.Path(destino)

    def _descargar(file_id, destino):
        if not pathlib.Path(destino).exists():
            import gdown

            gdown.download(id=file_id, output=destino, quiet=False)
        return pathlib.Path(destino)

    curso_setup = types.SimpleNamespace(
        en_colab=lambda: True,
        init=lambda *a, **k: pathlib.Path.cwd(),
        clonar_curso=_clonar,
        descargar_drive=_descargar,
    )

curso_setup.init()

<img src="https://medellin.unal.edu.co/eventos/panam2018/images/imagenes/organizan_2.png" width="40%">

# Procesamiento Digital de Imágenes

## Departamento de ingeniería eléctrica, electrónica y computación
## Sede Manizales

### Profesores Diego Pérez

In [ ]:
%%capture
# Instalar y actualizar bibliotecas necesarias
if curso_setup.en_colab():
    !pip install gdown
    !pip install roboflow
    !pip install ipywidgets
    !pip install ultralytics

In [3]:
# Importar bibliotecas a usar y deshabilitar WanDB

import os
import yaml
import gdown
import wandb
import shutil
import requests
from ultralytics import YOLO
from roboflow import Roboflow
from IPython.display import Image

os.environ['WANDB_DISABLED'] = 'true'
import wandb

In [ ]:
from roboflow import Roboflow
rf = Roboflow(api_key=os.environ["ROBOFLOW_API_KEY"])
project = rf.workspace("sergios-workspace-ajtlg").project("v9_ucrania_deforestacion")
version = project.version(1)
dataset = version.download("yolov8")

In [ ]:
# Entrenar YOLOv9 para detección de aisladores

# Carga de modelo preentrenado
model = YOLO('yolov8s-seg.pt')

# Entrenar el modelo
results = model.train(data='V9_Ucrania_Deforestación-1/data.yaml', epochs=300, imgsz=640, device=curso_setup.dispositivo())


In [ ]:
# Validar el modelo entrenado

# Carga de modelo entrenado
model = YOLO('runs/segment/train3/weights/best.pt')

# Realizar validación
validation_results = model.val(data='V9_Ucrania_Deforestación-1/data.yaml',
                               imgsz=640,
                               batch=16,
                               conf=0.5,
                               iou=0.6,
                               device=curso_setup.dispositivo())

In [ ]:
# Directorio de la imagen
imagen_dir = 'V9_Ucrania_Deforestación-1/test/images/img4_15_12_png.rf.e2f2366b47479183b4dc5ace29f8eba7.jpg'
# Realizar una inferencia
model.predict(imagen_dir, save=True, imgsz=640, conf=0.3, show_boxes=True, show_labels=True)
# Mostrar imagen sobre la cual se hizo inferencia
Image(filename="runs/segment/predict/"+imagen_dir.rsplit('/', 1)[-1])

In [ ]:
# Realizar validación
validation_results = model.val(data='Car-1/data.yaml',
                               imgsz=640,
                               batch=16,
                               conf=0.5,
                               iou=0.6,
                               device=curso_setup.dispositivo())

In [ ]:
project.version(dataset.version).deploy(model_type="yolov10", model_path=f"runs/detect/train")